In [1]:
#| default_exp restxl

In [18]:
#| hide
import nbdev; nbdev.nbdev_export()

In [3]:
!ln -s src rest/src

ln: failed to create symbolic link 'rest/src': File exists


In [4]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

In [5]:
#| export
from rest.core import init_instance, process_seq
singleton, model_path = init_instance()

In [6]:
#| export
import torch
import regex as re

In [7]:
#| export
seq_length = 512

from src.xl_wrapper import RuGPT3XL
model = RuGPT3XL.from_pretrained(
    "sberbank-ai/rugpt3xl",
    weights_path=f"./models/xl/{model_path}.model",
    deepspeed_config_path="src/deepspeed_config/gpt3_xl_sparse_2048.json",
    seq_len=seq_length
)

[W socket.cpp:426] [c10d] The server socket cannot be initialized on [::]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [::ffff:127.0.0.1]:6000 (errno: 97 - Address family not supported by protocol).
[W socket.cpp:601] [c10d] The client socket cannot be initialized to connect to [::ffff:127.0.0.1]:6000 (errno: 97 - Address family not supported by protocol).


> initializing model parallel with size 1
Use alternating sparse & dense attention layers


In [8]:
#| export
tokenizer = model.tokenizer
model.cuda()
model.eval();

In [9]:
sum(p.numel() for p in model.parameters())

1315737600

In [10]:
#| export
import deepspeed
ds_engine = deepspeed.init_inference(model,
                                 mp_size=1,
                                 dtype=torch.half,
                                 checkpoint=None,
                                 replace_method='auto',
                                 replace_with_kernel_inject=True)
model = ds_engine.module

[2022-11-11 14:11:39,098] [INFO] [logging.py:68:log_dist] [Rank -1] DeepSpeed info: version=0.7.5+28d4fdb, git-hash=28d4fdb, git-branch=master
[2022-11-11 14:11:39,099] [INFO] [logging.py:68:log_dist] [Rank -1] quantize_bits = 8 mlp_extra_grouping = False, quantize_groups = 1


In [11]:
#| export
def get_sample(prompt, length:int, num_samples:int, allow_linebreak:bool):
    encoded_prompt = tokenizer.encode(prompt, add_special_tokens=False, return_tensors="pt").cuda()
    encoded_prompt = encoded_prompt[:,length-(seq_length-1):]
    bad_words_ids = [tokenizer.encode('[')[0], tokenizer.encode('(')[0], tokenizer.encode('1\xa01')[1]]
    linebreak = tokenizer.encode("1\n1")[1]
    lb2 = tokenizer.encode("1 \n")[1]
    bad_words_ids += [] if allow_linebreak else [linebreak, lb2]
    bad_words_ids = [[b] for b in bad_words_ids] + [[linebreak,linebreak]]
    output_sequences = model.generate(
            input_ids=encoded_prompt,
            max_length=length + len(encoded_prompt[0]),
            temperature=1,
            top_k=0,
            top_p=0.9,
            do_sample=True,num_return_sequences=num_samples,
            bad_words_ids = bad_words_ids
        )
    if len(output_sequences.shape) > 2:
            output_sequences.squeeze_()
    generated_sequences = []
    for generated_sequence_idx, generated_sequence in enumerate(output_sequences):
        generated_sequence = generated_sequence.tolist()
        text = tokenizer.decode(generated_sequence, clean_up_tokenization_spaces=True)
        total_sequence = text[len(tokenizer.decode(encoded_prompt[0], clean_up_tokenization_spaces=True)) :]
        generated_sequences.append(total_sequence)

    return process_seq(generated_sequences)

In [15]:
%%time
get_sample(' - ты кто? \n - ', 50, 4, False)

not setting adaptive thresholding
CPU times: user 24.6 s, sys: 784 ms, total: 25.4 s
Wall time: 16.2 s


['.......... - бодро начал он и обмяк. - Принял... вторую. Теперь лежу. Мысли разные. В ногу, знаешь, отдача такая, что из винтовки вылетает. Так что, в общем, попал я на фронт.',
 '!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!! - Я Боженька! Я! Это я согреваю землю и облака! Я смотрю за тобой, и не могу не греть! Скажи-ка мне, Боже, сколько есть людей на свете?................................................',
 '~~~~~~~ ~~~~~~~ ~~~~~~~ ~~~~~~~ ~~~~~~~ ~~~~~~~ ~~~~~~~ ~~~~~~~ ~~~~~~~ ~~~~~~~ ',
 ' я? - раздался над болотом знакомый голос. - Я твой веселый друг: Никуша... Ты слышишь, скажи: Никуша!.. Но я - ответь: ты слышишь?.. - не отвечаю. Я слушаю...']

In [17]:
%%time
print(get_sample(' - ты кто?', 50, 1, True)[0])

not setting adaptive thresholding
 За что?.."
Милая,
Встань,
Выйди
За мною
К воротам,
Выйди
В сад
Раздумья.
Будешь
В гаданье
Гадалкой.

CPU times: user 2min 21s, sys: 3.88 s, total: 2min 25s
Wall time: 3.72 s
